# Step10b — Mapas Interactivos (Ecuador, 2015–2025)

**Proyecto:** EFCI Ecuador (ENSO Flood Chronicity Index)
**Fase:** 10 — Productos, sección B: 8 mapas interactivos HTML (Fase 0 §9)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

**Objetivo de este notebook:** generar la versión interactiva (HTML,
Leaflet vía `folium`) de los mismos 8 mapas estáticos de
`Step10a_Ecuador_Static_Maps.ipynb` — mismos datos y mismas clases
(quiebres naturales/Jenks), pero con zoom, paneo y un popup por cantón
al pasar el cursor (nombre, provincia, valor exacto).

**No recalcula nada**: reutiliza `mapa_master_canton.gpkg`, ya
construido y validado en Step10a.

**English:** Interactive maps (Leaflet/folium) for Ecuador, 2015–2025: the same thematic maps as the static atlas, with tooltips and legends.

## 0. Configuración del entorno

**English:** Environment setup.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper that creates folders when missing
from src.utilidades.io_utils import ensure_dir

# Tabular data
import pandas as pd
# Vector geodata
import geopandas as gpd
# Leaflet-based interactive maps
import folium
# Classification schemes (Natural Breaks / Jenks)
import mapclassify
# Linear colour scales for the legends
import branca.colormap as bcm

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10b_interactive_maps", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10b iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Output folder for the interactive HTML maps
maps_dir = ensure_dir(PROJECT_ROOT / "outputs" / "maps" / "interactivos")

# Read the master canton dataset built by the static-maps notebook
master = gpd.read_file(processed_dir / "mapa_master_canton.gpkg")
# Log the number of cantons loaded
logger.info("Dataset maestro cargado: %d cantones", len(master))

# Initial map centre (lat, lon)
ECUADOR_CENTER = [-1.5, -78.5]
# Initial zoom level
ECUADOR_ZOOM = 6

## 1. Función genérica: mapa interactivo coroplético (variable continua)

**English:** Generic function: interactive choropleth for a continuous variable.

In [ ]:
# Interactive choropleth for one continuous variable
def mapa_interactivo_continuo(gdf, columna, titulo, archivo, cmap_name="YlOrRd", k=5, fmt="{:,.0f}"):
    """Genera un mapa Leaflet (folium) coroplético para una variable continua,
    clasificada con quiebres naturales (Jenks) — misma clasificación que
    Step10a, para que el mapa interactivo y el estático sean coherentes.

    Args:
        gdf: GeoDataFrame maestro cantonal.
        columna: columna numérica a mapear.
        titulo: título mostrado en el mapa.
        archivo: nombre de archivo HTML de salida (sin extensión).
        cmap_name: paleta de branca.colormap (debe existir en linear.*).
        k: número de clases Jenks.
        fmt: formato de número para el popup.

    Returns:
        Ruta del archivo HTML generado.
    """
    # Natural Breaks classes, same scheme as the static maps
    clasificador = mapclassify.NaturalBreaks(gdf[columna], k=k)
    # Work on a copy
    gdf = gdf.copy()
    # Class index 0..k-1 per canton
    gdf["_clase"] = clasificador.yb  # índice de clase 0..k-1 por fila

    # branca nombra sus paletas con sufijo "_0N" (N = nº de colores, 03-11 según paleta)
    # Colour scale with k colours
    paleta = getattr(bcm.linear, f"{cmap_name}_{max(k, 3):02d}").scale(0, k - 1)
    # Colour of each class
    colores = {i: paleta(i) for i in range(k)}

    # Base map centred on Ecuador
    m = folium.Map(location=ECUADOR_CENTER, zoom_start=ECUADOR_ZOOM, tiles="cartodbpositron")

    # Draw the cantons with class colour and a tooltip
    folium.GeoJson(
        gdf,
        style_function=lambda feature: {
            "fillColor": colores[feature["properties"]["_clase"]],
            "color": "white", "weight": 0.5, "fillOpacity": 0.75,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["canton_name_official", "province_name_official", columna],
            aliases=["Cantón:", "Provincia:", titulo + ":"],
            localize=True, sticky=True,
        ),
    ).add_to(m)

    # HTML legend placed bottom-left
    leyenda_html = f'''
    <div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;
                background: white; padding: 10px; border: 1px solid #999; font-size: 12px;">
        <b>{titulo}</b><br>
    ''' + "".join(
        f'<span style="background:{colores[i]};width:12px;height:12px;display:inline-block;"></span> '
        f'{fmt.format(clasificador.bins[i-1] if i > 0 else gdf[columna].min())} - {fmt.format(clasificador.bins[i])}<br>'
        for i in range(k)
    ) + "</div>"
    # Attach the legend to the map
    m.get_root().html.add_child(folium.Element(leyenda_html))

    # Output HTML path
    out_path = maps_dir / f"{archivo}.html"
    # Save the interactive map
    m.save(str(out_path))
    # Log the generated map
    logger.info("Mapa interactivo generado: %s", out_path.name)
    # Return the file path
    return out_path

## 2. Los 6 mapas interactivos continuos

**English:** The continuous interactive maps.

In [ ]:
# Continuous maps: (column, title, file name, colour scale)
MAPAS_CONTINUOS = [
    ("frecuencia_inundaciones", "Frecuencia de Inundaciones", "01_frecuencia_inundaciones", "YlGnBu"),
    ("severity_index", "Índice de Severidad", "02_severidad", "OrRd"),
    ("frecuencia_enso", "Frecuencia en meses El Niño", "03_frecuencia_enso", "PuBu"),
    ("poblacion_total", "Población Afectada (máx. histórico)", "04_poblacion_afectada", "YlOrRd"),
    ("agricultura_total", "Pérdidas Agrícolas (máx. histórico, ha)", "05_perdidas_agricolas", "YlGn"),
    ("vivienda_total", "Pérdidas de Vivienda (máx. histórico)", "06_perdidas_vivienda", "Oranges"),
]

# Paths of generated maps
rutas_generadas = []
# Build one interactive map per variable
for columna, titulo, archivo, cmap in MAPAS_CONTINUOS:
    ruta = mapa_interactivo_continuo(master, columna, titulo, archivo, cmap_name=cmap)
    rutas_generadas.append(ruta)

## 3. Mapa interactivo de Cronicidad EFCI (clases ya fijadas en Fase 8)

**English:** Interactive chronicity map (classes fixed in the composite-index step).

In [ ]:
# Class names from the configuration
CLASS_LABELS = config["efci_composite"]["class_labels"]
# Sequential red palette, 5 classes
CLASS_COLORS = ["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"]
# Map each class to its colour
color_map_efci = dict(zip(CLASS_LABELS, CLASS_COLORS))

m = folium.Map(location=ECUADOR_CENTER, zoom_start=ECUADOR_ZOOM, tiles="cartodbpositron")
folium.GeoJson(
    master,
    style_function=lambda feature: {
        "fillColor": color_map_efci[feature["properties"]["clase_efci"]],
        "color": "white", "weight": 0.5, "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["canton_name_official", "province_name_official", "efci", "clase_efci"],
        aliases=["Cantón:", "Provincia:", "EFCI:", "Clase:"],
        localize=True, sticky=True,
    ),
).add_to(m)

leyenda_html = '<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: white; padding: 10px; border: 1px solid #999; font-size: 12px;"><b>Clase EFCI</b><br>' + "".join(
    f'<span style="background:{color_map_efci[c]};width:12px;height:12px;display:inline-block;"></span> {c}<br>' for c in CLASS_LABELS
) + "</div>"
m.get_root().html.add_child(folium.Element(leyenda_html))
m.save(str(maps_dir / "07_efci_cronicidad.html"))
logger.info("Mapa interactivo generado: 07_efci_cronicidad.html")

## 4. Mapa interactivo de Hotspots (Getis-Ord Gi*, Fase 9)

**English:** Interactive hotspot map (Getis-Ord Gi*).

In [ ]:
# Hotspot classes from hot to cold
HOTSPOT_ORDER = [
    "Punto Caliente 99%", "Punto Caliente 95%", "Punto Caliente 90%",
    "No significativo",
    "Punto Frío 90%", "Punto Frío 95%", "Punto Frío 99%",
]
# Diverging palette: red = hot, blue = cold
HOTSPOT_COLORS = ["#67000d", "#cb181d", "#fc9272", "#f0f0f0", "#9ecae1", "#3182bd", "#08306b"]
# Map each class to its colour
color_map_hotspot = dict(zip(HOTSPOT_ORDER, HOTSPOT_COLORS))

m = folium.Map(location=ECUADOR_CENTER, zoom_start=ECUADOR_ZOOM, tiles="cartodbpositron")
folium.GeoJson(
    master,
    style_function=lambda feature: {
        "fillColor": color_map_hotspot[feature["properties"]["gi_clase_efci"]],
        "color": "white", "weight": 0.5, "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["canton_name_official", "province_name_official", "gi_clase_efci"],
        aliases=["Cantón:", "Provincia:", "Clasificación Gi*:"],
        localize=True, sticky=True,
    ),
).add_to(m)

leyenda_html = '<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: white; padding: 10px; border: 1px solid #999; font-size: 11px;"><b>Clasificación Gi*</b><br>' + "".join(
    f'<span style="background:{color_map_hotspot[c]};width:12px;height:12px;display:inline-block;"></span> {c}<br>' for c in HOTSPOT_ORDER
) + "</div>"
m.get_root().html.add_child(folium.Element(leyenda_html))
m.save(str(maps_dir / "08_hotspots_gi_star.html"))
logger.info("Mapa interactivo generado: 08_hotspots_gi_star.html")

## 5. Validación

**English:** Validation.

In [ ]:
# List the generated HTML files
html_files = list(maps_dir.glob("*.html"))
# Check the expected number of maps
assert len(html_files) == 8, f"se esperaban 8 mapas interactivos, hay {len(html_files)}"
# Check no file is empty or corrupt
for f in html_files:
    assert f.stat().st_size > 1000, f"{f.name} parece vacío o corrupto"
logger.info("Fase 10b — validación OK. %d mapas interactivos en %s", len(html_files), maps_dir)
print("Fase 10b completa. Archivos:", [f.name for f in sorted(html_files)])